# Full Data Processing Pipeline (Kaggle Version)
Gộp 3 giai đoạn xử lý từ dữ liệu thô ra dataset chuẩn để train mô hình
- Giai đoạn 1: Chạy heuristics cắt clip (tạm bỏ qua LLM để dễ chạy ngay).
- Giai đoạn 2: Tạo Fake User Interactions.
- Giai đoạn 3: Phân chia thư mục thành cấu trúc `ready-data` chuẩn bị cho bước Train/Demo.

In [ ]:
!pip install -q pydub orjson tqdm pandas numpy

## 1. Setup Thư Viện & Đường Dẫn

In [ ]:
import os
import re
import json
import time
import logging
import random
import warnings
import shutil
import difflib
from pathlib import Path
from typing import Dict, Any, List
from collections import defaultdict

import numpy as np
import pandas as pd
import orjson
from tqdm.auto import tqdm

# Cho xử lý Audio
try:
    from pydub import AudioSegment
except ImportError:
    pass

# ==========================================
# CẤU HÌNH HỆ THỐNG VÀ PATHS
# ==========================================

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# => Cập nhật chính xác tên paths trên Kaggle theo như ảnh chụp workspace của bạn
BASE_DIR = Path("/kaggle/input/datasets/nquanggnguyn/podtok/data") 
DATA_DIR = BASE_DIR

# Ở ảnh bạn chụp, transcript nằm ở 1 dataset riêng được add vào /kaggle/input/transcript/transcripts
TRANSCRIPT_DIR = Path("/kaggle/input/datasets/nquanggnguyn/transcript/transcripts")
SPLIT_AUDIO_DIR = DATA_DIR / "raw_audio_split"
METADATA_PATH = DATA_DIR / "metadata.json"


OUTPUT_DIR = Path("/kaggle/working/outputs")
FINAL_CLIPS_DIR = OUTPUT_DIR / "final_clips"
FINAL_METADATA_DIR = OUTPUT_DIR / "final_metadata"
CLIPS_DIR = OUTPUT_DIR / "clips"

FINAL_CLIPS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_METADATA_DIR.mkdir(parents=True, exist_ok=True)
CLIPS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Thư mục Data (Input): {DATA_DIR}")
logger.info(f"Thư mục Transcript: {TRANSCRIPT_DIR}")
logger.info(f"Thư mục Audio nguồn: {SPLIT_AUDIO_DIR}")
logger.info(f"Thư mục Outputs (Working): {OUTPUT_DIR}")

# ==========================================
# CẤU HÌNH HEURISTICS & LLM
# ==========================================
IDEAL_MIN_SEC = 25.0
IDEAL_MAX_SEC = 45.0
MIN_CLIP_SEC = 15.0
MAX_CLIP_SEC = 55.0
MAX_CANDIDATES_PER_EPISODE = 20
TOP_K_CLIPS_PER_EPISODE = 5

USE_LLM = False # Để False tạm thời cho dễ chạy. Bật True nếu muốn dùng Local LLM
USE_4BIT = True
USE_BFLOAT16 = False
USE_FLASH_ATTENTION = False
LLM_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct" # Đã thay từ bản Qwen1.5 thành Qwen2.5
LLM_BATCH_SIZE = 4
MAX_NEW_TOKENS = 128
LLM_GROUP_SIZE = 5
LLM_TOP_PER_GROUP = 2

## 2. Hàm Tiện Ích

In [ ]:
def read_json(path: Path):
    if not path.exists(): return {}
    with open(path, "rb") as f:
        return orjson.loads(f.read())

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists(): return rows
    with open(path, "rb") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            rows.append(orjson.loads(line))
    return rows

def write_jsonl(records: List[Dict[str, Any]], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        for row in records:
            f.write(orjson.dumps(row))
            f.write(b"\n")

def clean_text(text: Any) -> str:
    if text is None: return ""
    text = str(text)
    return re.sub(r"\s+", " ", text.strip())

def normalize_episode_id_from_name(name: str) -> str:
    base = Path(name).stem
    return base.replace("_split", "")

def transcript_path_to_episode_id(path: Path) -> str:
    return normalize_episode_id_from_name(path.name)

def audio_path_to_episode_id(path: Path) -> str:
    return normalize_episode_id_from_name(path.name)

## 3. Giai Đoạn 1: Cắt Clip (Placeholder)

In [ ]:
logger.info("=== BẮT ĐẦU GIAI ĐOẠN 1: TẠO CLIP ===")
item_metadata_final_path = OUTPUT_DIR / "item_metadata_final.jsonl"

metadata_rows = []
if METADATA_PATH.exists():
    if Path(METADATA_PATH).suffix == '.json':
        try:
             metadata_rows = orjson.loads(METADATA_PATH.read_bytes())
             if not isinstance(metadata_rows, list): metadata_rows = [metadata_rows]
        except Exception:
             metadata_rows = read_jsonl(METADATA_PATH)
    else:
        metadata_rows = read_jsonl(METADATA_PATH)
        
metadata_map = {row.get("_id", row.get("id", "")): row for row in metadata_rows}

transcript_paths = sorted(TRANSCRIPT_DIR.glob("*.json"))
audio_paths = sorted(SPLIT_AUDIO_DIR.glob("*.mp3"))
audio_map = {audio_path_to_episode_id(p): p for p in audio_paths}

episode_index = []
for tpath in transcript_paths:
    episode_id = transcript_path_to_episode_id(tpath)
    meta = metadata_map.get(episode_id, {})
    episode_index.append({
        "episode_id": episode_id,
        "transcript_path": str(tpath),
        "split_audio_path": str(audio_map.get(episode_id, "")),
        "title": meta.get("title"),
        "host": meta.get("host"),
        "keyword": meta.get("keyword")
    })

write_jsonl(episode_index, OUTPUT_DIR / "episode_index.jsonl")

def generate_placeholder_final_items():
    logger.info("Tự động tạo một số items mẫu dựa vào audio/transcripts map.")
    final_items = []
    for idx, (ep_id, p) in enumerate(audio_map.items()):
        final_items.append({
            "clip_id": f"clip_{ep_id}_{idx}",
            "item_id": idx + 1,
            "episode_id": ep_id,
            "title": f"Title {ep_id}",
            "host": f"Host {ep_id}",
            "keyword": f"Keyword {ep_id}",
            "duration_sec": 30.5,
            "clip_audio_path": f"outputs/clips/{ep_id}/clip_{ep_id}_{idx}.mp3",
            "source_audio_file": str(p),
            "transcript_text": f"Nội dung giả lập cho {ep_id}..."
        })
    write_jsonl(final_items, item_metadata_final_path)

if not item_metadata_final_path.exists():
    generate_placeholder_final_items()
else:
    logger.info("Item metadata đã tồn tại, skip auto generate.")

## 4. Giai Đoạn 2: Mock User Interactions

In [ ]:
logger.info("=== BẮT ĐẦU GIAI ĐOẠN 2: MOCK USER INTERACTIONS ===")

MOCK_OUTPUTS_DIR = Path("/kaggle/working/mock_outputs")
MOCK_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

NUM_USERS = 50 
SESSION_DAY_SPAN = 30
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
RNG = random.Random(SEED)

raw_items = read_jsonl(item_metadata_final_path)
prepared_items = []

for idx, row in enumerate(raw_items, start=1):
    clip_id = clean_text(row.get("clip_id"))
    if not clip_id: continue
    prepared = dict(row)
    prepared["item_id"] = idx
    prepared["keyword_norm"] = clean_text(row.get("keyword")).lower()
    prepared["host_norm"] = clean_text(row.get("host")).lower()
    prepared_items.append(prepared)

if prepared_items:
    user_profiles = []
    keyword_pool = list(set([r["keyword_norm"] for r in prepared_items if r["keyword_norm"]]))
    host_pool = list(set([r["host_norm"] for r in prepared_items if r["host_norm"]]))

    for user_idx in range(1, NUM_USERS + 1):
        fav_keywords = RNG.sample(keyword_pool, k=min(3, len(keyword_pool))) if keyword_pool else []
        fav_hosts = RNG.sample(host_pool, k=min(2, len(host_pool))) if host_pool else []
        user_profiles.append({
            "user_id": f"user_{user_idx:05d}",
            "fav_keywords": fav_keywords,
            "fav_hosts": fav_hosts,
        })
    write_jsonl(user_profiles, MOCK_OUTPUTS_DIR / "mock_user_profiles.jsonl")
    
    events = []
    event_counter = 1
    base_time = pd.Timestamp("2026-04-01T07:00:00Z")

    for profile in user_profiles:
        num_sessions = RNG.randint(2, 5)
        user_time = base_time + pd.Timedelta(days=RNG.randint(0, SESSION_DAY_SPAN - 1))
        for session_idx in range(1, num_sessions + 1):
            session_id = f"sess_{profile['user_id']}_{session_idx:04d}"
            session_length = RNG.randint(5, 15)
            cursor_time = user_time
            for position in range(1, session_length + 1):
                item_row = RNG.choice(prepared_items)
                watch_label = 1 if RNG.random() > 0.5 else 0
                events.append({
                    "event_id": f"evt_{event_counter:09d}",
                    "user_id": profile["user_id"],
                    "session_id": session_id,
                    "event_time": cursor_time.isoformat().replace("+00:00", "Z"),
                    "clip_id": item_row["clip_id"],
                    "item_id": item_row["item_id"],
                    "position_in_session": position,
                    "watch_label": watch_label,
                    "watch_time_sec": item_row.get("duration_sec", 30.0) * RNG.uniform(0.1, 1.0)
                })
                event_counter += 1
                cursor_time += pd.Timedelta(seconds=RNG.randint(10, 60))
            user_time = cursor_time + pd.Timedelta(hours=RNG.randint(5, 24))

    interactions_df = pd.DataFrame(events)
    interactions_df.to_json(MOCK_OUTPUTS_DIR / "user_interactions.jsonl", orient="records", lines=True)
    interactions_df.to_parquet(MOCK_OUTPUTS_DIR / "user_interactions.parquet", index=False)
    logger.info(f"Đã tạo {len(events)} sự kiện mock thành công.")

## 5. Giai Đoạn 3: Assemble Ready-Data

In [ ]:
logger.info("=== BẮT ĐẦU GIAI ĐOẠN 3: ASSEMBLE READY-DATA ===")

def copy_file(src: Path, dst: Path):
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

READY_DATA_ROOT = Path("/kaggle/working/ready-data")
READY_DATA_ROOT.mkdir(parents=True, exist_ok=True)
(READY_DATA_ROOT / "catalog").mkdir(parents=True, exist_ok=True)
(READY_DATA_ROOT / "interactions/mock").mkdir(parents=True, exist_ok=True)
(READY_DATA_ROOT / "demo/feed").mkdir(parents=True, exist_ok=True)

copy_file(item_metadata_final_path, READY_DATA_ROOT / "catalog/item_metadata_final.jsonl")
copy_file(MOCK_OUTPUTS_DIR / "mock_user_profiles.jsonl", READY_DATA_ROOT / "interactions/mock/user_profiles.jsonl")
copy_file(MOCK_OUTPUTS_DIR / "user_interactions.jsonl", READY_DATA_ROOT / "interactions/mock/user_interactions.jsonl")
copy_file(MOCK_OUTPUTS_DIR / "user_interactions.parquet", READY_DATA_ROOT / "interactions/mock/user_interactions.parquet")

try:
    if item_metadata_final_path.exists():
        items_for_feed = read_jsonl(item_metadata_final_path)
        with open(READY_DATA_ROOT / "demo/feed/feed_items.json", "wb") as f:
            f.write(orjson.dumps({"items": items_for_feed[:20]}, option=orjson.OPT_INDENT_2))
except Exception as e:
    logger.error(f"Lỗi: {e}")

with open(READY_DATA_ROOT / "manifest.json", "wb") as f:
    f.write(orjson.dumps({
         "package_name": "ready-data", 
         "version": "v1", 
         "source": "full_data_processing_pipeline_kaggle_notebook"
    }, option=orjson.OPT_INDENT_2))

logger.info(f"Hoàn thành! Bạn có thể tải file từ: {READY_DATA_ROOT}")